# 🏅 Medallion Architecture Overview

The Medallion Architecture structures data sequentially to guarantee data quality and governance across the Lakehouse.

In [0]:
# Verify the personal container is reachable
_container = dbutils.widgets.get("container")
_storage_account = dbutils.widgets.get("storage_account")
_secret_scope = dbutils.widgets.get("secret_scope")

print(_container)
print(_storage_account)
print(_secret_scope)

spark.conf.set(f"fs.azure.account.auth.type.{_storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{_storage_account}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{_storage_account}.dfs.core.windows.net", dbutils.secrets.get(scope=_secret_scope, key="sp-databricks-adls-appid"))
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{_storage_account}.dfs.core.windows.net", dbutils.secrets.get(scope=_secret_scope, key="sp-databricks-adls-appkey"))
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{_storage_account}.dfs.core.windows.net", f"https://login.microsoftonline.com/{dbutils.secrets.get(scope=_secret_scope, key='tenant-id')}/oauth2/token")

display(dbutils.fs.ls(f"abfss://{_container}@{_storage_account}.dfs.core.windows.net/"))

In [0]:
# dbutils.fs.put(
#     f"abfss://{_container}@{_storage_account}.dfs.core.windows.net/sample.txt",
#     "Hello from Databricks!",
#     overwrite=True
# )

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `joshuandegwa_external_location`
URL 'abfss://joshuandegwa7447@dlspl21databricks.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `databricks_uc_connector`)
COMMENT 'External location for student Joshua Ndegwa';


In [0]:
%sql
-- Confirm Unity Catalog is created. 
DESCRIBE EXTERNAL LOCATION `joshuandegwa_external_location`;

In [0]:
%sql
-- Ensure you are using the General Catalog
USE CATALOG dbr_dev; 

-- 1. Create Bronze Schema
CREATE SCHEMA IF NOT EXISTS joshuandegwa_bronze
MANAGED LOCATION 'abfss://joshuandegwa7447@dlspl21databricks.dfs.core.windows.net/bronze'
COMMENT 'Bronze layer for raw ingested data for Joshua Ndegwa';

-- 2. Create Silver Schema
CREATE SCHEMA IF NOT EXISTS joshuandegwa_silver
MANAGED LOCATION 'abfss://joshuandegwa7447@dlspl21databricks.dfs.core.windows.net/silver'
COMMENT 'Silver layer for cleaned/transformed data for Joshua Ndegwa';

-- 3. Create Gold Schema
CREATE SCHEMA IF NOT EXISTS joshuandegwa_gold
MANAGED LOCATION 'abfss://joshuandegwa7447@dlspl21databricks.dfs.core.windows.net/gold'
COMMENT 'Gold layer for business-ready aggregated data for Joshua Ndegwa';

In [0]:
%sql
SHOW SCHEMAS LIKE 'joshuandegwa_*';

In [0]:
%sql
DESCRIBE SCHEMA EXTENDED joshuandegwa_bronze;

In [0]:
# List scopes to ensure my secret scope is available
# SecretScope(name='joshuaSecretScope')
dbutils.secrets.listScopes()

In [0]:
%sql
USE CATALOG dbr_dev;
USE SCHEMA joshuandegwa_bronze;

CREATE EXTERNAL VOLUME IF NOT EXISTS raw_files
LOCATION 'abfss://joshuandegwa7447@dlspl21databricks.dfs.core.windows.net/raw';

In [0]:
# # 1. Fetch widget values with leading underscores
# _container = dbutils.widgets.get("container")
# _storage_account = dbutils.widgets.get("storage_account")
# _secret_scope = dbutils.widgets.get("secret_scope")

# Hardcoded catalog & user schema parameters
catalog_name = "dbr_dev"
login_prefix = "joshuandegwa"
schema_name = f"{login_prefix}_bronze"
volume_name = "raw_files"

# 2. Build storage base URL and Volume location
base_storage_url = f"abfss://{_container}@{_storage_account}.dfs.core.windows.net"
volume_location = f"{base_storage_url}/raw"

# 3. Create External Volume via SQL
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {schema_name}")

spark.sql(f"""
CREATE EXTERNAL VOLUME IF NOT EXISTS {volume_name}
LOCATION '{volume_location}'
COMMENT 'External volume for raw landing files';
""")

print(f"External Volume '{volume_name}' ready at: {volume_location}")

##### Bronze ingestion: load files into your <login>_bronze schema as Delta, idempotently (no duplicate loads), with metadata columns (source filename, ingestion timestamp, load date). 

In [0]:
# %pip install -q kagglehub
# dbutils.library.restartPython() # Restart python to use installed kagglehub

In [0]:
import os
import kagglehub

# Path to the dataset inside the external volume
catalog_name = "dbr_dev"
login_prefix = "joshuandegwa"
schema_name = f"{login_prefix}_bronze"
volume_name = "raw_files"
dataset_folder = "banking"
target_volume_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/banking"


# 1. Download dataset from Kaggle to driver local disk
dataset_path = kagglehub.dataset_download("vivekmali1436/banking-transactions-dataset")
print(f"Downloaded files to local driver path: {dataset_path}")

# 2. Ensure banking directory exists inside the volume
dbutils.fs.mkdirs(target_volume_path)

# 3. Copy files to Volume
for file_name in os.listdir(dataset_path):
    local_file_path = os.path.join(dataset_path, file_name)
    
    if os.path.isfile(local_file_path):
        dest_file_path = f"{target_volume_path}/{file_name}"
        dbutils.fs.cp(f"file://{local_file_path}", dest_file_path)
        print(f" Transferred {file_name} -> {dest_file_path}")

print(f"\n Files landed successfully at: {target_volume_path}")

In [0]:
volume_base_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/{dataset_folder}"
storage_base_url = f"abfss://{_container}@{_storage_account}.dfs.core.windows.net"

spark.sql(f"USE CATALOG `{catalog_name}`")
spark.sql(f"USE SCHEMA `{schema_name}`")

In [0]:
import os

# Find all CSV files to ingest
landing_files = [f for f in os.listdir(volume_base_path) if f.endswith(".csv")]
print(f"Found {len(landing_files)} CSV files ready for ingestion:")
for file in landing_files:
    print(f" - {file}")

In [0]:
from pyspark.sql.functions import col, current_timestamp, current_date

# Cell 3: Streaming Ingestion & Metadata Tagging (UC-Compatible)
for file in landing_files:
    entity_name = os.path.splitext(file)[0]
    target_table_fqn = f"`{catalog_name}`.`{schema_name}`.`bronze_{entity_name}`"
    
    file_source_path = f"{volume_base_path}/{file}"
    checkpoint_path = f"{storage_base_url}/_checkpoints/bronze/{entity_name}"
    schema_location = f"{storage_base_url}/_schemas/bronze/{entity_name}"
    
    print(f"Ingesting: {file} -> {target_table_fqn}")

    df_stream = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_location)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("header", "true")
        .load(file_source_path)
        # --- METADATA COLUMNS (UNITY CATALOG COMPATIBLE) ---
        .withColumn("_source_filename", col("_metadata.file_path")) 
        .withColumn("_ingested_timestamp", current_timestamp())
        .withColumn("_load_date", current_date())
    )

    query = (
        df_stream.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .toTable(target_table_fqn)
    )

    query.awaitTermination()
    print(f" Completed: {target_table_fqn}\n")

print("All files ingested into Bronze layer successfully!")